<a href="https://colab.research.google.com/github/10kshaodow/Deep-Learning-Lab-2/blob/main/Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib as plt
import torch
import torch.nn.functional as f
from datasets import load_dataset
import os
import math
import random
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#https://huggingface.co/datasets/wmt/wmt14
ds = load_dataset("wmt/wmt14", "de-en")
train_ds = ds["train"]
test_ds = ds["test"]
val_ds = ds["validation"]

tokenizer_train_ds = train_ds.select(range(50000)) # tokenizer training time needed to be reduced in order to proceed past this cell

tokenizer = None
tokenizer_path = "/content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer/my_trained_tokenizer.json"
 #this path should be different for you
if not os.path.exists(tokenizer_path):
    print(f"Tokenizer file not found at {tokenizer_path}. Starting training...")
    # https://huggingface.co/docs/transformers/en/fast_tokenizers
    tokenizer_core = Tokenizer(BPE(unk_token="[UNK]"))

    trainer = BpeTrainer(
        vocab_size=37000,
        special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
    )

    tokenizer_core.pre_tokenizer = Whitespace()

    def tokenizer_generator(ds):
        for row in ds:
            yield row["translation"]["en"]
            yield row["translation"]["de"]

    try:
        # Ensure the directory exists before saving
        tokenizer_dir = os.path.dirname(tokenizer_path)
        os.makedirs(tokenizer_dir, exist_ok=True)
        if os.path.exists(tokenizer_dir):
            print(f"Directory '{tokenizer_dir}' ensured to exist.")
        else:
            print(f"Warning: Directory '{tokenizer_dir}' could not be created or is not accessible.")

        tokenizer_core.train_from_iterator(tokenizer_generator(tokenizer_train_ds), trainer)
        tokenizer_core.save(tokenizer_path)
        print(f"Tokenizer trained and saved to {tokenizer_path}.")

        print("File exists after saving:", os.path.exists(tokenizer_path))
        print("Files in tokenizer directory:", os.listdir(tokenizer_dir))
    except KeyboardInterrupt:
        print("Tokenizer training interrupted by user. The tokenizer file might not be saved or complete.")
        raise # Re-raise to indicate that execution stopped
    except Exception as e:
        print(f"An unexpected error occurred during tokenizer training: {e}")
        raise



    # Now, initialize PreTrainedTokenizerFast from the trained tokenizer_core
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_core)
    tokenizer.pad_token = "[PAD]"
    tokenizer.bos_token = "[BOS]"
    tokenizer.eos_token = "[EOS]"

else:
    print(f"Tokenizer file found at {tokenizer_path}. Loading existing tokenizer...")
    tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
    tokenizer.pad_token = "[PAD]"
    tokenizer.bos_token = "[BOS]"
    tokenizer.eos_token = "[EOS]"
    print("Tokenizer loaded successfully.")


max_len = 128
batch_size = 128


def preprocess(batch): #takes each dataset example and converts it into numbers.

    src = batch["translation"]["de"]
    tgt = batch["translation"]["en"]

    model_inputs = tokenizer(src, max_length=max_len, truncation=True)
    labels = tokenizer(text_target=tgt, max_length=max_len, truncation=True)

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# https://huggingface.co/docs/datasets/en/use_dataset
train_tokens = ds["train"].map(
    preprocess,
    remove_columns=ds["train"].column_names,
    num_proc=4
)

test_tokens = ds["test"].map(
    preprocess,
    remove_columns=ds["test"].column_names,
    num_proc=4
)

val_tokens = ds["validation"].map(
    preprocess,
    remove_columns=ds["validation"].column_names,
    num_proc=4
)


collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=tokenizer.pad_token_id
)


train_loader = DataLoader(
    train_tokens,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collator
)

test_loader = DataLoader(
    test_tokens,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    val_tokens,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collator
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer file not found at /content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer/my_trained_tokenizer.json. Starting training...
Directory '/content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer' ensured to exist.
Tokenizer trained and saved to /content/drive/MyDrive/CMPE-679-Labs/Lab2-Transformer/my_trained_tokenizer.json.
File exists after saving: True
Files in tokenizer directory: ['my_trained_tokenizer.json']


Map (num_proc=4):   0%|          | 0/4508785 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3003 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/3000 [00:00<?, ? examples/s]

#Positional Encoding

In [4]:
import torch.nn as nn
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Create a matrix of shape (max_len, d_model) filled with zeros
        pe = torch.zeros(max_len, d_model)

        # Create a vector of positions (0, 1, 2, ..., max_len-1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Calculate the division term for the sine and cosine functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Apply sine to even indices and cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add a batch dimension and register as a buffer (not a trainable parameter)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x shape: (batch_size, seq_len, d_model)
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]